In [ ]:
!pip install google-play-scraper vaderSentiment

In [ ]:
import requests
from google_play_scraper import reviews, Sort
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd
from datetime import datetime, timezone
import time

In [ ]:
# ---------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------

GPLAY_APP_ID    = "com.bumble.app"
APPSTORE_APP_ID = "930441707"
GPLAY_COUNT     = 2000
APPSTORE_PAGES  = 10   # max 10 pages x 50 reviews = 500 reviews
OUTPUT_GPLAY    = "bumble_google_play.csv"
OUTPUT_APPSTORE = "bumble_app_store.csv"

print("Configuration loaded.")
print("Note: App store data will reflect recent reviews only due to API limitations.")

Configuration loaded.
Note: App store data will reflect recent reviews only due to API limitations.


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def score_sentiment(text):
    """Run VADER and return scores + label."""
    scores   = analyzer.polarity_scores(text)
    compound = scores["compound"]
    label    = "positive" if compound >= 0.05 else "negative" if compound <= -0.05 else "neutral"
    return {
        "vader_compound":  round(compound, 4),
        "vader_positive":  round(scores["pos"], 4),
        "vader_negative":  round(scores["neg"], 4),
        "vader_neutral":   round(scores["neu"], 4),
        "sentiment_label": label
    }

def build_record(source, body, date_str, rating, thumbs_up):
    """Build a standardised record dict for a single review."""
    return {
        "source":        source,
        "original_date": date_str,
        "comment_date":  None,
        "text":          body,
        "comment":       body,
        "subreddit":     None,
        "post_title":    None,
        "post_score":    None,
        "comment_score": None,
        "rating":        rating,
        "thumbs_up":     thumbs_up,
        **score_sentiment(body)
    }

print("Helper functions ready.")

Helper functions ready.


In [ ]:
# ---------------------------------------------------------------
# GOOGLE PLAY SCRAPER
# ---------------------------------------------------------------
print("Scraping Google Play reviews for Bumble...")

gplay_results, _ = reviews(
    GPLAY_APP_ID,
    lang="en",
    country="us",
    sort=Sort.NEWEST,
    count=GPLAY_COUNT,
    filter_score_with=None
)

gplay_data = []

for r in gplay_results:
    # r["at"] is a timezone-aware datetime object from google_play_scraper
    review_date = r["at"]
    if review_date.tzinfo is None:
        review_date = review_date.replace(tzinfo=timezone.utc)

    body = r.get("content", "")
    if not body or len(body.strip()) < 10:
        continue

    gplay_data.append(build_record(
        source    = "google_play",
        body      = body,
        date_str  = review_date.strftime("%Y-%m-%d"),
        rating    = r.get("score"),
        thumbs_up = r.get("thumbsUpCount", 0)
    ))

df_gplay = pd.DataFrame(gplay_data)
print(f"Google Play: {len(df_gplay)} reviews collected")
print(f"Date range:  {df_gplay['original_date'].min()} to {df_gplay['original_date'].max()}")

Scraping Google Play reviews for Bumble...
Google Play: 1725 reviews collected
Date range:  2026-03-20 to 2026-05-17


In [ ]:
# ---------------------------------------------------------------
# APP STORE SCRAPER (iTunes RSS)
# Note: Returns ~500 most recent reviews only
# ---------------------------------------------------------------
print("Scraping App Store reviews for Bumble via iTunes RSS...")

DATE_FORMATS = [
    "%Y-%m-%dT%H:%M:%S-07:00",
    "%Y-%m-%dT%H:%M:%S-08:00",
    "%Y-%m-%dT%H:%M:%S+00:00",
    "%Y-%m-%dT%H:%M:%SZ",
]

def parse_appstore_date(raw):
    """Try multiple date formats, return datetime or None."""
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(raw, fmt).replace(tzinfo=timezone.utc)
        except:
            pass
    try:
        return datetime.fromisoformat(raw.replace("Z", "+00:00"))
    except:
        return None

base_url      = f"https://itunes.apple.com/us/rss/customerreviews/page={{page}}/id={APPSTORE_APP_ID}/sortby=mostrecent/json"
appstore_data = []

for page in range(1, APPSTORE_PAGES + 1):
    try:
        response = requests.get(base_url.format(page=page), timeout=15)
        feed     = response.json().get("feed", {})
        entries  = feed.get("entry", [])

        if not entries:
            print(f"  No more entries at page {page}, stopping.")
            break

        page_kept = 0
        for entry in entries:
            if "im:rating" not in entry:
                continue

            body = entry.get("content", {}).get("label", "")
            if not body or len(body.strip()) < 10:
                continue

            raw_date    = entry.get("updated", {}).get("label", "")
            review_date = parse_appstore_date(raw_date)
            date_str    = review_date.strftime("%Y-%m-%d") if review_date else ""
            rating      = int(entry.get("im:rating", {}).get("label", 0))

            appstore_data.append(build_record(
                source    = "app_store",
                body      = body,
                date_str  = date_str,
                rating    = rating,
                thumbs_up = None
            ))
            page_kept += 1

        print(f"  Page {page}: {page_kept} reviews kept")
        time.sleep(0.5)

    except Exception as e:
        print(f"  ERROR on page {page}: {e}")
        continue

df_appstore = pd.DataFrame(appstore_data)
print(f"\nApp Store: {len(df_appstore)} reviews collected")
if len(df_appstore) > 0:
    print(f"Date range: {df_appstore['original_date'].min()} to {df_appstore['original_date'].max()}")

Scraping App Store reviews for Bumble via iTunes RSS...
  Page 1: 49 reviews kept
  Page 2: 49 reviews kept
  Page 3: 50 reviews kept
  Page 4: 50 reviews kept
  Page 5: 48 reviews kept
  Page 6: 50 reviews kept
  Page 7: 48 reviews kept
  Page 8: 48 reviews kept
  Page 9: 47 reviews kept
  Page 10: 50 reviews kept

App Store: 489 reviews collected
Date range: 2026-03-19 to 2026-05-17


In [ ]:
# ---------------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------------
for label, df in [("Google Play", df_gplay), ("App Store", df_appstore)]:
    print(f"\n{'='*50}")
    print(f"{label} SUMMARY")
    print(f"{'='*50}")
    if len(df) == 0:
        print("No data collected.")
        continue
    print(f"Total reviews:   {len(df)}")
    print(f"Date range:      {df['original_date'].min()} to {df['original_date'].max()}")
    print(f"Avg star rating: {df['rating'].mean():.2f} / 5")
    print(f"Avg compound:    {df['vader_compound'].mean():.3f}")
    print(f"\nSentiment breakdown:")
    print(df["sentiment_label"].value_counts(normalize=True).mul(100).round(1).to_string())
    print(f"\nStar rating distribution:")
    print(df["rating"].value_counts().sort_index().to_string())


Google Play SUMMARY
Total reviews:   1725
Date range:      2026-03-20 to 2026-05-17
Avg star rating: 1.66 / 5
Avg compound:    -0.087

Sentiment breakdown:
sentiment_label
negative    50.1
positive    34.3
neutral     15.6

Star rating distribution:
rating
1    1272
2     152
3      85
4      55
5     161

App Store SUMMARY
Total reviews:   489
Date range:      2026-03-19 to 2026-05-17
Avg star rating: 1.50 / 5
Avg compound:    -0.053

Sentiment breakdown:
sentiment_label
negative    49.7
positive    39.3
neutral     11.0

Star rating distribution:
rating
1    380
2     45
3     25
4      8
5     31


In [ ]:
df_gplay.to_csv(OUTPUT_GPLAY, index=False)
df_appstore.to_csv(OUTPUT_APPSTORE, index=False)
print(f"Saved {len(df_gplay)} rows to {OUTPUT_GPLAY}")
print(f"Saved {len(df_appstore)} rows to {OUTPUT_APPSTORE}")
print(f"Columns: {list(df_gplay.columns)}")

Saved 1725 rows to bumble_google_play.csv
Saved 489 rows to bumble_app_store.csv
Columns: ['source', 'original_date', 'comment_date', 'text', 'comment', 'subreddit', 'post_title', 'post_score', 'comment_score', 'rating', 'thumbs_up', 'vader_compound', 'vader_positive', 'vader_negative', 'vader_neutral', 'sentiment_label']


In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_GPLAY)
    files.download(OUTPUT_APPSTORE)
    print("Downloads triggered.")
except ImportError:
    print("Not in Colab — files saved locally.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads triggered.
